# Fine-tune a local claims classifier (QLoRA, free Colab T4)

End-to-end run of the `local-llm-actuary` fine-tuning recipe on a free Colab GPU.

**Runtime > Change runtime type > T4 GPU** before running.

**Data rule:** this notebook is acceptable on hosted compute only because every training example is synthetic. With real data, train on hardware you control.

In [ ]:
!nvidia-smi


In [ ]:
!pip install -q unsloth transformers trl datasets peft bitsandbytes


In [ ]:
!git clone https://github.com/globebyte/local-llm-actuary.git
%cd local-llm-actuary


## 1. Prepare the dataset (160 train / 40 held-out test)

In [ ]:
!python finetune/01_prepare_dataset.py


## 2. Train (QLoRA on Qwen3-4B) and export GGUF

Minutes on a T4 for this dataset.

In [ ]:
!python finetune/02_finetune_qlora.py


## 3. Download the artefacts

Fetch the zip, then on your own machine, from a clone of this repository:

```
unzip finetuned_model.zip
python finetune/01_prepare_dataset.py
ollama create claims-classifier -f finetune/outputs/gguf_gguf/Modelfile
python finetune/03_evaluate.py --models qwen3:8b claims-classifier
```

Step 1 is rerun locally because the split files are not committed. Its fixed seed reproduces the identical 40-note test set, which is what makes the comparison against the prompted baseline meaningful.

In [ ]:
from google.colab import files
import glob
for f in glob.glob('finetune/outputs/gguf_gguf/*'):
    print(f)
# files.download(<path>)  # uncomment per file, or zip the folder:
!zip -r -q finetuned_model.zip finetune/outputs/gguf_gguf
files.download('finetuned_model.zip')
